In [ ]:
pygame.display

pygame.draw

pygame.image

pygame.transform

pygame.font

pygame.mixer

pygame.time

pygame.event

pygame.key

pygame.mouse

pygame.sprite

pygame.surface.Surface

pygame.locals


### 1. Show the Window and Exit

In [2]:
import pygame
pygame.init()

screen = pygame.display.set_mode((640, 480))
pygame.display.set_caption("My Pygame Demo")

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:04:14.120 python[28774:5685889] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:04:14.120 python[28774:5685889] +[IMKInputSession subclass]: chose IMKInputSession_Modern


### 2.Targeting the Plan

In [2]:
import pygame
import random

# 初始化
pygame.init()
WIDTH, HEIGHT = 480, 640
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("打飞机小游戏")
clock = pygame.time.Clock()
FPS = 60

# 颜色
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)

# 字体
font = pygame.font.SysFont("Arial", 24)

# 玩家飞机类
class Player(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        self.image = pygame.Surface((40, 60))
        self.image.fill((0, 128, 255))
        self.rect = self.image.get_rect()
        self.rect.centerx = WIDTH // 2
        self.rect.bottom = HEIGHT - 10
        self.speed = 5

    def update(self, keys):
        if keys[pygame.K_LEFT] and self.rect.left > 0:
            self.rect.x -= self.speed
        if keys[pygame.K_RIGHT] and self.rect.right < WIDTH:
            self.rect.x += self.speed

# 敌机类
class Enemy(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        self.image = pygame.Surface((30, 30))
        self.image.fill((255, 0, 0))
        self.rect = self.image.get_rect()
        self.rect.x = random.randint(0, WIDTH - self.rect.width)
        self.rect.y = random.randint(-100, -40)
        self.speedy = random.randint(2, 6)

    def update(self):
        self.rect.y += self.speedy
        if self.rect.top > HEIGHT:
            self.rect.x = random.randint(0, WIDTH - self.rect.width)
            self.rect.y = random.randint(-100, -40)

# 子弹类
class Bullet(pygame.sprite.Sprite):
    def __init__(self, x, y):
        super().__init__()
        self.image = pygame.Surface((5, 10))
        self.image.fill((255, 255, 0))
        self.rect = self.image.get_rect()
        self.rect.centerx = x
        self.rect.bottom = y
        self.speedy = -10

    def update(self):
        self.rect.y += self.speedy
        if self.rect.bottom < 0:
            self.kill()

# 生成对象
player = Player()
player_group = pygame.sprite.Group(player)
enemy_group = pygame.sprite.Group()
bullet_group = pygame.sprite.Group()

for _ in range(5):
    enemy_group.add(Enemy())

score = 0
running = True

# 游戏主循环
while running:
    clock.tick(FPS)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        # 发射子弹
        if event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
            bullet_group.add(Bullet(player.rect.centerx, player.rect.top))

    keys = pygame.key.get_pressed()
    player.update(keys)
    enemy_group.update()
    bullet_group.update()

    # 碰撞检测
    hits = pygame.sprite.groupcollide(enemy_group, bullet_group, True, True)
    for _ in hits:
        score += 10
        enemy_group.add(Enemy())

    # 绘制画面
    screen.fill(BLACK)
    player_group.draw(screen)
    enemy_group.draw(screen)
    bullet_group.draw(screen)

    score_text = font.render(f"Score: {score}", True, WHITE)
    screen.blit(score_text, (10, 10))

    pygame.display.flip()

pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:04:35.811 python[28880:5689627] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:04:35.811 python[28880:5689627] +[IMKInputSession subclass]: chose IMKInputSession_Modern
2025-04-19 23:04:37.313 python[28880:5689627] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


### 3. Maine Menu of Game Design

In [2]:
import pygame
import sys

pygame.init()
WIDTH, HEIGHT = 600, 450
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("游戏主界面系统")

# ✅ 替换为 macOS 系统内置中文字体（建议也可以换成 PingFang.ttc 或思源宋体）
font = pygame.font.Font("/System/Library/Fonts/STHeiti Light.ttc", 36)
clock = pygame.time.Clock()

WHITE = (255, 255, 255)
GRAY = (200, 200, 200)
DARK_GRAY = (100, 100, 100)
BLACK = (0, 0, 0)
BLUE = (0, 120, 255)
GREEN = (0, 200, 100)

MAIN_MENU = "main_menu"
LEVEL_SELECT = "level_select"
SETTINGS = "settings"
TUTORIAL = "tutorial"
current_scene = MAIN_MENU

class Button:
    def __init__(self, text, x, y, width, height, action=None):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.color = GRAY
        self.hover_color = DARK_GRAY
        self.action = action

    def draw(self, surface):
        mouse_pos = pygame.mouse.get_pos()
        color = self.hover_color if self.rect.collidepoint(mouse_pos) else self.color
        pygame.draw.rect(surface, color, self.rect)
        text_surf = font.render(self.text, True, BLACK)
        text_rect = text_surf.get_rect(center=self.rect.center)
        surface.blit(text_surf, text_rect)

    def is_clicked(self, event):
        return event.type == pygame.MOUSEBUTTONDOWN and self.rect.collidepoint(event.pos)

def switch_scene(scene_name):
    global current_scene
    current_scene = scene_name

def exit_game():
    pygame.quit()
    sys.exit()

# 按钮集合（中文显示支持 ✅）
main_menu_buttons = [
    Button("开始游戏", 220, 100, 160, 50, lambda: switch_scene(LEVEL_SELECT)),
    Button("设置", 220, 170, 160, 50, lambda: switch_scene(SETTINGS)),
    Button("教学", 220, 240, 160, 50, lambda: switch_scene(TUTORIAL)),
    Button("退出", 220, 310, 160, 50, lambda: exit_game())
]

level_buttons = [
    Button("关卡 1", 220, 120, 160, 50),
    Button("关卡 2", 220, 190, 160, 50),
    Button("返回主菜单", 220, 260, 160, 50, lambda: switch_scene(MAIN_MENU))
]

settings_buttons = [
    Button("音乐：开", 200, 120, 200, 50),
    Button("难度：普通", 200, 190, 200, 50),
    Button("返回主菜单", 200, 260, 200, 50, lambda: switch_scene(MAIN_MENU))
]

tutorial_buttons = [
    Button("点击按钮开始动画", 180, 160, 240, 50),
    Button("返回主菜单", 200, 250, 200, 50, lambda: switch_scene(MAIN_MENU))
]

show_animation = False
circle_x = 100

while True:
    screen.fill(WHITE)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            exit_game()
        elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            switch_scene(MAIN_MENU)
        active_buttons = []
        if current_scene == MAIN_MENU:
            active_buttons = main_menu_buttons
        elif current_scene == LEVEL_SELECT:
            active_buttons = level_buttons
        elif current_scene == SETTINGS:
            active_buttons = settings_buttons
        elif current_scene == TUTORIAL:
            active_buttons = tutorial_buttons

        for btn in active_buttons:
            if btn.is_clicked(event):
                if btn.action:
                    btn.action()
                if current_scene == TUTORIAL and "动画" in btn.text:
                    show_animation = True
                    circle_x = 100

    # 渲染界面
    if current_scene == MAIN_MENU:
        screen.blit(font.render("主菜单", True, BLUE), (250, 30))
        for btn in main_menu_buttons:
            btn.draw(screen)

    elif current_scene == LEVEL_SELECT:
        screen.blit(font.render("选择关卡", True, BLUE), (230, 30))
        for btn in level_buttons:
            btn.draw(screen)

    elif current_scene == SETTINGS:
        screen.blit(font.render("设置", True, BLUE), (260, 30))
        for btn in settings_buttons:
            btn.draw(screen)

    elif current_scene == TUTORIAL:
        screen.blit(font.render("教学页面", True, BLUE), (230, 30))
        for btn in tutorial_buttons:
            btn.draw(screen)
        if show_animation:
            pygame.draw.circle(screen, GREEN, (circle_x, 350), 20)
            circle_x += 3
            if circle_x > WIDTH:
                show_animation = False

    pygame.display.flip()
    clock.tick(60)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:12:45.182 python[29193:5701530] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:12:45.182 python[29193:5701530] +[IMKInputSession subclass]: chose IMKInputSession_Modern


SystemExit: 

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 4.图形绘制

In [2]:
import pygame
import random
import sys

pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("图形绘制 + 粒子系统演示")
clock = pygame.time.Clock()

# ✅ 加载 macOS 中文字体
font = pygame.font.Font("/System/Library/Fonts/STHeiti Light.ttc", 24)

WHITE = (255, 255, 255)
GRAY = (220, 220, 220)
BLACK = (0, 0, 0)

# 粒子类
class Particle:
    def __init__(self, x, y):
        self.x = x + random.randint(-10, 10)
        self.y = y + random.randint(-10, 10)
        self.radius = random.randint(3, 7)
        self.color = [random.randint(100, 255) for _ in range(3)]
        self.dy = random.uniform(-1.5, -0.5)
        self.alpha = 255

    def update(self):
        self.y += self.dy
        self.alpha -= 3  # 渐隐
        if self.alpha < 0:
            self.alpha = 0

    def draw(self, surface):
        s = pygame.Surface((self.radius * 2, self.radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(s, (*self.color, self.alpha), (self.radius, self.radius), self.radius)
        surface.blit(s, (self.x - self.radius, self.y - self.radius))

particles = []

def draw_grid():
    for x in range(0, WIDTH, 40):
        pygame.draw.line(screen, GRAY, (x, 0), (x, HEIGHT))
    for y in range(0, HEIGHT, 40):
        pygame.draw.line(screen, GRAY, (0, y), (WIDTH, y))

def draw_shapes():
    pygame.draw.rect(screen, (255, 100, 100), (50, 50, 100, 80))  # 矩形
    pygame.draw.circle(screen, (100, 255, 100), (200, 90), 40)     # 圆形
    pygame.draw.polygon(screen, (100, 100, 255), [(300, 50), (350, 130), (250, 130)])  # 三角形
    pygame.draw.line(screen, BLACK, (400, 50), (500, 130), 3)      # 线条

def main():
    running = True
    while running:
        screen.fill(WHITE)
        draw_grid()
        draw_shapes()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.MOUSEBUTTONDOWN:
                mx, my = pygame.mouse.get_pos()
                for _ in range(20):  # 生成一团粒子
                    particles.append(Particle(mx, my))

        for p in particles[:]:
            p.update()
            p.draw(screen)
            if p.alpha == 0:
                particles.remove(p)

        # ✅ 中文说明标签
        label = font.render("点击屏幕生成粒子，展示图形绘制（矩形、圆、多边形、线条）", True, BLACK)
        screen.blit(label, (20, HEIGHT - 40))

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    sys.exit()

main()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:23:06.864 python[29656:5717142] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:23:06.864 python[29656:5717142] +[IMKInputSession subclass]: chose IMKInputSession_Modern


SystemExit: 

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 5.输入框系统

In [2]:
import pygame
import sys
import os

pygame.init()
WIDTH, HEIGHT = 800, 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("📝 中文输入框系统示例")
clock = pygame.time.Clock()

WHITE = (255, 255, 255)
GRAY = (200, 200, 200)
BLUE = (100, 150, 255)
BLACK = (0, 0, 0)

# ✅ 中文字体支持（尝试加载本地思源黑体）
font_path = "NotoSansSC-Regular.otf"  # 请确保此字体文件存在于项目目录
if not os.path.exists(font_path):
    font_path = pygame.font.match_font("msyh") or pygame.font.get_default_font()
font = pygame.font.Font(font_path, 24)

# -------------------- 输入框类 --------------------
class InputBox:
    def __init__(self, x, y, w, h, label="", is_password=False):
        self.rect = pygame.Rect(x, y, w, h)
        self.color = GRAY
        self.text = ""
        self.active = False
        self.label = label
        self.is_password = is_password

    def handle_event(self, e):
        if e.type == pygame.MOUSEBUTTONDOWN:
            self.active = self.rect.collidepoint(e.pos)
        if e.type == pygame.KEYDOWN and self.active:
            if e.key == pygame.K_RETURN:
                print(f"{self.label}: {self.text}")
                self.text = ""
            elif e.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                self.text += e.unicode  # ✅ 可接收中文字符

    def draw(self, surface):
        display_text = "*" * len(self.text) if self.is_password else self.text
        txt_surface = font.render(display_text, True, BLACK)
        label_surface = font.render(self.label, True, BLACK)
        pygame.draw.rect(surface, BLUE if self.active else self.color, self.rect, 2)
        surface.blit(label_surface, (self.rect.x, self.rect.y - 30))
        surface.blit(txt_surface, (self.rect.x + 5, self.rect.y + 5))

# -------------------- 创建输入框 --------------------
input_boxes = [
    InputBox(200, 100, 400, 40, label="用户名"),
    InputBox(200, 180, 400, 40, label="密码", is_password=True)
]

# -------------------- 主循环 --------------------
while True:
    screen.fill(WHITE)

    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            pygame.quit()
            sys.exit()
        for box in input_boxes:
            box.handle_event(e)

    for box in input_boxes:
        box.draw(screen)

    pygame.display.flip()
    clock.tick(60)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:32:26.270 python[30026:5730315] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:32:26.270 python[30026:5730315] +[IMKInputSession subclass]: chose IMKInputSession_Modern


SystemExit: 

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 粒子爆炸模拟器

In [4]:
import pygame
import random
import math
import sys

pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("🔥 粒子爆炸模拟器")
clock = pygame.time.Clock()

# 粒子类
class Particle:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        angle = random.uniform(0, 2 * math.pi)
        speed = random.uniform(2, 6)
        self.dx = math.cos(angle) * speed
        self.dy = math.sin(angle) * speed
        self.radius = random.randint(3, 6)
        self.color = [random.randint(150, 255) for _ in range(3)]
        self.alpha = 255
        self.gravity = 0.15
        self.friction = 0.98
        self.shrink = random.uniform(0.93, 0.98)

    def update(self):
        self.dy += self.gravity
        self.x += self.dx
        self.y += self.dy
        self.dx *= self.friction
        self.dy *= self.friction
        self.radius *= self.shrink
        self.alpha -= 4
        if self.alpha < 0:
            self.alpha = 0

    def is_dead(self):
        return self.alpha <= 0 or self.radius <= 0.5

    def draw(self, surface):
        s = pygame.Surface((self.radius * 2, self.radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(s, (*self.color, self.alpha), (self.radius, self.radius), int(self.radius))
        surface.blit(s, (self.x - self.radius, self.y - self.radius))

particles = []

# 主循环
def main():
    running = True
    while running:
        screen.fill((10, 10, 10))  # 背景深灰色

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.MOUSEBUTTONDOWN:
                mx, my = pygame.mouse.get_pos()
                for _ in range(100):
                    particles.append(Particle(mx, my))

        for p in particles[:]:
            p.update()
            p.draw(screen)
            if p.is_dead():
                particles.remove(p)

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    sys.exit()

main()

SystemExit: 

### 6.物理弹跳系统+实时UI

In [2]:
import pygame
import sys
import random
import math

pygame.init()
WIDTH, HEIGHT = 1000, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("实时UI + 物理弹跳系统")
clock = pygame.time.Clock()
font = pygame.font.SysFont("Arial", 18)

WHITE, GRAY, RED, GREEN, BLUE, BLACK = (255,255,255), (200,200,200), (220,60,60), (60,220,60), (60,60,220), (0,0,0)

# -------------------- 滑动条 --------------------
class Slider:
    def __init__(self, x, y, w, min_val, max_val, default):
        self.rect = pygame.Rect(x, y, w, 10)
        self.knob_x = x + (default - min_val) / (max_val - min_val) * w
        self.min = min_val
        self.max = max_val
        self.val = default
        self.dragging = False

    def draw(self, surface):
        pygame.draw.rect(surface, GRAY, self.rect)
        pygame.draw.circle(surface, RED, (int(self.knob_x), self.rect.centery), 8)
        label = font.render(f"Gravity: {self.val:.2f}", True, BLACK)
        surface.blit(label, (self.rect.x, self.rect.y - 20))

    def handle_event(self, e):
        if e.type == pygame.MOUSEBUTTONDOWN and pygame.Rect(self.knob_x-10, self.rect.y-5, 20, 20).collidepoint(e.pos):
            self.dragging = True
        elif e.type == pygame.MOUSEBUTTONUP:
            self.dragging = False
        elif e.type == pygame.MOUSEMOTION and self.dragging:
            self.knob_x = max(self.rect.x, min(e.pos[0], self.rect.right))
            pct = (self.knob_x - self.rect.x) / self.rect.width
            self.val = self.min + pct * (self.max - self.min)

# -------------------- 小球类 --------------------
class Ball:
    def __init__(self):
        self.x = random.randint(100, 900)
        self.y = 50
        self.dx = random.uniform(-2, 2)
        self.dy = 0
        self.radius = random.randint(10, 15)
        self.color = [random.randint(120,255) for _ in range(3)]

    def update(self, gravity):
        self.dy += gravity
        self.x += self.dx
        self.y += self.dy

        if self.y + self.radius > HEIGHT - 100:
            self.y = HEIGHT - 100 - self.radius
            self.dy *= -0.7
        if self.x - self.radius < 0 or self.x + self.radius > WIDTH:
            self.dx *= -1

    def draw(self, surface):
        pygame.draw.circle(surface, self.color, (int(self.x), int(self.y)), self.radius)

# -------------------- 波形图 --------------------
waves = []
def draw_waveform():
    waves.append(math.sin(pygame.time.get_ticks()*0.005) + random.uniform(-0.2, 0.2))
    if len(waves) > 300: waves.pop(0)
    step = 2
    points = [(i*step, HEIGHT-50 + v*30) for i,v in enumerate(waves)]
    pygame.draw.rect(screen, GRAY, (0, HEIGHT-100, WIDTH, 100))
    if len(points) > 2:
        pygame.draw.lines(screen, BLUE, False, points, 2)

# -------------------- 雷达图 --------------------
def draw_radar(ball):
    cx, cy = WIDTH-150, 150
    pygame.draw.circle(screen, GRAY, (cx, cy), 60, 1)
    pygame.draw.circle(screen, GRAY, (cx, cy), 40, 1)
    pygame.draw.circle(screen, GRAY, (cx, cy), 20, 1)
    speed = min(1.0, (ball.dx**2 + ball.dy**2)**0.5 / 10)
    bounce = min(1.0, abs(ball.dy) / 10)
    height = min(1.0, 1 - ball.y / HEIGHT)
    angles = [0, 120, 240]
    values = [speed, bounce, height]
    points = []
    for i, val in enumerate(values):
        angle = math.radians(angles[i])
        r = val * 60
        x = cx + r * math.cos(angle)
        y = cy + r * math.sin(angle)
        points.append((x, y))
    pygame.draw.polygon(screen, GREEN, points)
    pygame.draw.polygon(screen, BLACK, points, 1)
    screen.blit(font.render("Radar (Speed/Bounce/Height)", True, BLACK), (cx - 70, cy + 70))

# -------------------- 主程序 --------------------
balls = [Ball() for _ in range(5)]
g_slider = Slider(50, HEIGHT-130, 300, 0.1, 1.5, 0.5)

while True:
    screen.fill(WHITE)
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            pygame.quit(); sys.exit()
        g_slider.handle_event(e)

    # 更新小球并绘制
    for b in balls:
        b.update(g_slider.val)
        b.draw(screen)

    g_slider.draw(screen)
    draw_waveform()
    draw_radar(balls[0])

    pygame.display.flip()
    clock.tick(60)


pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:30:07.222 python[29868:5725017] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:30:07.222 python[29868:5725017] +[IMKInputSession subclass]: chose IMKInputSession_Modern


SystemExit: 

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 7.计时器

In [2]:
import pygame
import sys

pygame.init()
WIDTH, HEIGHT = 600, 300
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Time Counter")
font = pygame.font.SysFont("Arial", 36)
clock = pygame.time.Clock()

WHITE = (255, 255, 255)
RED = (220, 60, 60)
BLACK = (0, 0, 0)

# 正计时起始点
start_ticks = pygame.time.get_ticks()

# 倒计时秒数
countdown_seconds = 10
countdown_active = True
countdown_finished = False

# -------------------- 主循环 --------------------
while True:
    screen.fill(WHITE)

    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

    # 当前已用时间（正计时器）
    elapsed_ms = pygame.time.get_ticks() - start_ticks
    elapsed_seconds = elapsed_ms // 1000
    timer_text = font.render(f"Counter: {elapsed_seconds} Second", True, BLACK)
    screen.blit(timer_text, (50, 50))

    # 倒计时器逻辑
    if countdown_active:
        remain = max(0, countdown_seconds - elapsed_seconds)
        countdown_text = font.render(f"Counter: {remain} Second", True, RED if remain <= 3 else BLACK)
        screen.blit(countdown_text, (50, 120))
        if remain == 0 and not countdown_finished:
            countdown_finished = True
            countdown_active = False
    else:
        if countdown_finished:
            done_text = font.render("Time is Up！", True, RED)
            screen.blit(done_text, (50, 180))

    pygame.display.flip()
    clock.tick(60)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:37:31.045 python[30271:5738938] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:37:31.045 python[30271:5738938] +[IMKInputSession subclass]: chose IMKInputSession_Modern


SystemExit: 

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 8.小型AI追踪

In [2]:
import pygame
import sys
import math

pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption(" AI：Following the Hero")
clock = pygame.time.Clock()
font = pygame.font.SysFont("Arial", 24)

WHITE = (255, 255, 255)
BLUE = (80, 150, 255)
RED = (255, 80, 80)
BLACK = (0, 0, 0)

# -------------------- 主角类 --------------------
class Player:
    def __init__(self):
        self.x = WIDTH // 2
        self.y = HEIGHT // 2
        self.speed = 4
        self.radius = 20

    def handle_keys(self):
        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]: self.x -= self.speed
        if keys[pygame.K_RIGHT]: self.x += self.speed
        if keys[pygame.K_UP]: self.y -= self.speed
        if keys[pygame.K_DOWN]: self.y += self.speed
        self.x = max(self.radius, min(WIDTH - self.radius, self.x))
        self.y = max(self.radius, min(HEIGHT - self.radius, self.y))

    def draw(self, surface):
        pygame.draw.circle(surface, BLUE, (self.x, self.y), self.radius)

# -------------------- 敌人类（带追踪 AI） --------------------
class Enemy:
    def __init__(self):
        self.x = 100
        self.y = 100
        self.speed = 2.5
        self.radius = 15

    def update(self, target_x, target_y):
        dx = target_x - self.x
        dy = target_y - self.y
        distance = math.hypot(dx, dy)
        if distance > 1:
            dx /= distance
            dy /= distance
            self.x += dx * self.speed
            self.y += dy * self.speed

    def draw(self, surface):
        pygame.draw.circle(surface, RED, (int(self.x), int(self.y)), self.radius)

# -------------------- 实例化 --------------------
player = Player()
enemy = Enemy()

# -------------------- 主循环 --------------------
while True:
    screen.fill(WHITE)

    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

    player.handle_keys()
    enemy.update(player.x, player.y)

    player.draw(screen)
    enemy.draw(screen)

    # 文字提示
    text = font.render("方向键控制主角，敌人自动追踪", True, BLACK)
    screen.blit(text, (20, 20))

    pygame.display.flip()
    clock.tick(60)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:36:44.011 python[30248:5738264] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:36:44.011 python[30248:5738264] +[IMKInputSession subclass]: chose IMKInputSession_Modern


SystemExit: 

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 9.登陆后聊天系统

In [2]:
import pygame
import sys
import os

pygame.init()
WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Log in to enter the chatting room")
clock = pygame.time.Clock()

WHITE = (255, 255, 255)
GRAY = (200, 200, 200)
BLUE = (100, 150, 255)
BLACK = (0, 0, 0)
LIGHTGRAY = (230, 230, 230)

font_path = "NotoSansSC-Regular.otf"
if not os.path.exists(font_path):
    font_path = pygame.font.match_font("msyh") or pygame.font.get_default_font()
font = pygame.font.Font(font_path, 24)

# -------------------- 输入框类 --------------------
class InputBox:
    def __init__(self, x, y, w, h, label="", is_password=False):
        self.rect = pygame.Rect(x, y, w, h)
        self.color = GRAY
        self.text = ""
        self.active = False
        self.label = label
        self.is_password = is_password

    def handle_event(self, e):
        if e.type == pygame.MOUSEBUTTONDOWN:
            self.active = self.rect.collidepoint(e.pos)
        if e.type == pygame.KEYDOWN and self.active:
            if e.key == pygame.K_RETURN:
                return self.text
            elif e.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            else:
                self.text += e.unicode

    def draw(self, surface):
        display_text = "*" * len(self.text) if self.is_password else self.text
        txt_surface = font.render(display_text, True, BLACK)
        label_surface = font.render(self.label, True, BLACK)
        pygame.draw.rect(surface, BLUE if self.active else self.color, self.rect, 2)
        surface.blit(label_surface, (self.rect.x, self.rect.y - 30))
        surface.blit(txt_surface, (self.rect.x + 5, self.rect.y + 5))

# -------------------- 按钮类 --------------------
class Button:
    def __init__(self, text, x, y, w, h, callback):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = text
        self.callback = callback

    def draw(self, surface):
        pygame.draw.rect(surface, BLUE, self.rect)
        txt = font.render(self.text, True, WHITE)
        surface.blit(txt, txt.get_rect(center=self.rect.center))

    def handle_event(self, e):
        if e.type == pygame.MOUSEBUTTONDOWN and self.rect.collidepoint(e.pos):
            self.callback()

# -------------------- 状态变量 --------------------
logged_in = False
chat_history = []

# -------------------- 登录页面元素 --------------------
username_box = InputBox(250, 180, 300, 40, label="user name")
password_box = InputBox(250, 260, 300, 40, label="password", is_password=True)

def try_login():
    global logged_in
    if username_box.text and password_box.text:
        logged_in = True

login_button = Button("login", 330, 330, 140, 40, try_login)

# -------------------- 聊天界面元素 --------------------
chat_input = InputBox(50, 520, 700, 40, label="DM")

def send_chat():
    global chat_history
    if chat_input.text:
        chat_history.append(f"{username_box.text}：{chat_input.text}")
        chat_input.text = ""

send_button = Button("send", 680, 460, 70, 40, send_chat)

# -------------------- 主循环 --------------------
while True:
    screen.fill(WHITE)
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            pygame.quit()
            sys.exit()

        if not logged_in:
            username_box.handle_event(e)
            password_box.handle_event(e)
            login_button.handle_event(e)
        else:
            chat_input.handle_event(e)
            send_button.handle_event(e)

    if not logged_in:
        title = font.render("please input the user name and password", True, BLACK)
        screen.blit(title, (240, 100))
        username_box.draw(screen)
        password_box.draw(screen)
        login_button.draw(screen)
    else:
        pygame.draw.rect(screen, LIGHTGRAY, (50, 50, 700, 390))
        for i, msg in enumerate(chat_history[-12:]):
            msg_surface = font.render(msg, True, BLACK)
            screen.blit(msg_surface, (60, 60 + i * 30))
        chat_input.draw(screen)
        send_button.draw(screen)

    pygame.display.flip()
    clock.tick(60)


pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


2025-04-19 23:41:47.076 python[30431:5744533] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-19 23:41:47.076 python[30431:5744533] +[IMKInputSession subclass]: chose IMKInputSession_Modern
2025-04-19 23:41:48.881 python[30431:5744533] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


SystemExit: 

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
